In [ ]:
import numpy as np
import timeit

# Size of test data
shape = (1000000, 3)
data = np.random.rand(*shape)

# Class using __array__
class ArrayWrapper:
    def __init__(self, arr):
        self.arr = arr

    def __array__(self, dtype=None):
        return np.array(self.arr, dtype=dtype)

# Class using __array_interface__
class InterfaceWrapper:
    def __init__(self, arr):
        self._arr = arr

    @property
    def __array_interface__(self):
        return self._arr.__array_interface__


# Create instances
array_wrapper = ArrayWrapper(data)
interface_wrapper = InterfaceWrapper(data)

# Define timing code
setup_code = "from __main__ import np, array_wrapper, interface_wrapper"

# Measure time to convert using np.array (will call __array__)
# time_array_wrapper = timeit.timeit("np.array(array_wrapper)", setup=setup_code, number=100)
#
# # Measure time to convert using np.asarray (should use __array_interface__ if defined)
# time_interface_wrapper = timeit.timeit("np.asarray(interface_wrapper)", setup=setup_code, number=100)

# (time_array_wrapper, time_interface_wrapper)

In [ ]:
# Measure time to convert using np.array (will call __array__)
time_array_wrapper = timeit.timeit("np.linalg.norm(np.array(array_wrapper))", setup=setup_code, number=100)
# Measure time to convert using np.asarray (should use __array_interface__ if defined)
time_interface_wrapper = timeit.timeit("np.linalg.norm(np.asarray(interface_wrapper))", setup=setup_code, number=100)
(time_array_wrapper, time_interface_wrapper)

In [ ]:
import numpy as np

from pchandler.v2.base_arrays import BaseArray
from pchandler.v2.geometry.core import PointCloudData
from pchandler.v2.geometry.core_pydantic import BasePointCloud
from pchandler.v2.geometry.coordinates import OptimisedCartesianCoordinates
from pchandler.v2.geometry.scalar_fields_pydantic import ScalarFieldManager

import timeit
data = np.random.rand(1000000, 3)
time_point_cloud_data = timeit.timeit("PointCloudData(xyz=data)", setup=setup_code, number=100, globals=globals())
time_base_array = timeit.timeit("BaseArray(arr=data)", setup=setup_code, number=100, globals=globals())
time_opt_cart_array = timeit.timeit("OptimisedCartesianCoordinates(arr=data)", setup=setup_code, number=100, globals=globals())
time_base_pcd = timeit.timeit("BasePointCloud(arr=data, sfm=ScalarFieldManager())", setup=setup_code, number=100, globals=globals())

(time_point_cloud_data, time_base_array, time_opt_cart_array, time_base_pcd)

In [ ]:
a = PointCloudData(xyz=data)
b = BaseArray(arr=data.astype(np.float32))
c = BasePointCloud(arr=data.astype(np.float32), sfm=ScalarFieldManager())
d = OptimisedCartesianCoordinates(arr=data)

In [ ]:
%%timeit -r 5 -n 100
a = PointCloudData(xyz=data)

Point Cloud Initialisation - 14.4 ms ± 1.99 ms per loop (mean ± std. dev. of 5 runs, 100 loops each)

In [ ]:
%%timeit -r 5 -n 100
a_ = np.linalg.norm(a.xyz) * 2.3 + 3.2 - 1.22 * 2323
a__ = np.sin(a.xyz)

Point Cloud Computation - 7.33 ms ± 436 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [ ]:
%%timeit -r 5 -n 100
b = BaseArray(arr=data.astype(np.float32))

Base Array Initialisation - 4.8 ms ± 286 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [ ]:
%%timeit -r 5 -n 100
b_ = np.linalg.norm(b.arr) * 2.3 + 3.2 - 1.22 *2323
b__ = np.sin(b.arr)

Base Array Computation - 7.17 ms ± 238 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)7.38 ms ± 263 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [ ]:
%%timeit -r 5 -n 100
c = BasePointCloud(arr=data.astype(np.float32), sfm=ScalarFieldManager())

In [ ]:
%%timeit -r 5 -n 100
c_ = np.linalg.norm(c.arr) * 2.3 + 3.2 - 1.22 *2323
f = np.sin(c.arr)

In [ ]:
ind = np.zeros(b.shape[0], dtype=np.bool_)
ind[np.random.randint(0, b.shape[0], 10000)] = True
np.sum(ind)

In [ ]:
%%timeit -r 5 -n 100
mask = a._convert_indexing_to_mask(ind)
da = a.sample(ind)

2.12 ms ± 16.9 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [ ]:
%%timeit -r 5 -n 100
db = b.sample(ind)

3.38 ms ± 53.3 μs per loop (mean ± std. dev. of 5 runs, 1,000 loops each)

In [ ]:
%%timeit -r 5 -n 100
a = b[ind]
b = b[~ind]

In [ ]:
%%timeit -r 5 -n 100
a, c = b[ind], b[~ind]

In [ ]:
from pydantic import BaseModel, ConfigDict
from functools import cached_property

class A(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    @cached_property
    def test(self):
        return 10

In [ ]:
a = A()
print(hasattr(a, 'test'))

In [ ]:
print(hasattr(a.__dict__, 'test'))

In [ ]:
a.test


In [ ]:
print(hasattr(a.__dict__, 'test'))

In [ ]:
b = a.model_copy()

In [ ]:
a.__dict__

In [ ]:
b.__dict__

In [ ]:
id(a.test)

In [ ]:
id(b.test)

In [ ]:
import numpy as np
from pchandler.v2.base_arrays import BaseArray

a = BaseArray(arr = np.random.rand(10,3))

_What is the expected **default** behaviour of the base array?_ -> Consider cases such as scalar fields, coordinates, transform matrices

#### Case 1: b=a
- b is a pointer to the object a
- Changes to b also changes a

In [ ]:
b = a
print('b=a')
print(f'    Obj Ref : {id(b)     == id(a)     = }')
print(f'    Arr Ref : {id(b.arr) == id(a.arr) = }')
print(f'    View    : {b.base is None         = }')

#### Case 2: b = a.copy()
- shallow copy
- Object `b` is "different" to `a` but their `arr` attributes are equal (same `__array_interface__`)

In [ ]:
b = a.copy()
print('b=a.copy()')
print(f'    Obj Ref : {id(b)     == id(a)     = }')
print(f'    Arr Ref : {id(b.arr) == id(a.arr) = }')
print(f'    View    : {b.base is None         = }')

#### Case 3: b = a.copy(deep=True)
- deep copy
- Object `b` is a deepcopy of `a` so that no IDs of any object in the new object match the old.
- Ensures changes to `b` don't change `a`

In [ ]:
b = a.copy(deep=True)
print('b=a.copy(deep=True)')
print(f'    Obj Ref : {id(b)     != id(a)     = }')
print(f'    Arr Ref : {id(b.arr) == id(a.arr) = }')
print(f'    View    : {b.base is None         = }')

#### Case 4(a): a.view()
- View access to array

In [ ]:
b = a.view()
print('b=a.view()')
print(f'    Obj Ref : {id(b)     != id(a)        = }')
print(f'    Arr Ref : {id(b.arr) != id(a.arr)    = }')
print(f'    View    : {b.base is not None        = }')
print(f'    View    : {np.allclose(b.base, a.arr)= }')

#### Case 4: b = a.copy(view=True)
- View of array
- Object b is a shallow copy but the array attribute is actually a view back to the original object.
- Changes to b would change a. But this should be done for "Read-Only" objects to enable data manipulation

In [10]:
b = a.copy(view=True)
print('b=a.copy(view=True)')
print(f'    Obj Ref : {id(b)     != id(a)     = }')
print(f'    Arr Ref : {id(b.arr) != id(a.arr) = }')
print(f'    View    : {b.arr.base is not None = }')

b=a
    Pointer: id(b)==id(a)         = False
    Array  : id(b.arr)==id(a.arr) = False
    View   : b.base is None       = True


### Slicing and Advanced Indexing => View Casting and Copying

Is there a desire to support views?
 - e.g. avoid copying and support copy in place functions?

In numpy, slicing and ellipses return a **view**. Whilst advanced indexing (boolean masks, numpy array of ints) produce a **copy**.
```python
a = np.ones(100)
b = a[:]        # This is a view
b = a[1, 2, 3]  # This is a copy

Current behaviour converted slices to masks and therefore a copy was always created.

##### Pt Cloud
For most point cloud sampling I assume they are not ordered. Therefore slicing will be seldom used. Indexing will predominantly be via the advanced indexing -> integer or conditional boolean mask.

#### Image or scalar fields
Is there a need for slicing where you are performing an analysis on one part of the image?
`arr[min_row:max_row, min_col:max_col]` -> Sliding window analysis
Other GPU based operations which benefit from passing views rather than having to copy data?

In [13]:
a = np.random.rand(100)
print(f'{id(a)=}')
b = a.view()
print(f'{id(b)=}')
c = a[:]
print(f'Is view: {a.base is not None}')
print(f'Is view: {b.base is not None}')
print(f'Is view: {c.base is not None}')

id(a)=140110297280560
id(b)=140110297278352
Is view: False
Is view: True
Is view: True


In [16]:
# This is a copy
mask = np.ones_like(a, dtype=np.bool_)
d = a[mask]
print(f'Same ID: {id(d)==id(a)}')
print(f'Is view: {d.base is not None}')

Same ID: False
Is view: False
